# Dependências

In [ ]:
# Install required Google Cloud packages (commented out as these are typically one-time setup commands)
!pip install gcloud
!gcloud auth application-default login

# Import necessary Python libraries
import pandas as pd                # Data manipulation and analysis
import numpy as np                 # Numerical computing
import time                        # Time-related functions
import os                          # Operating system interfaces
import pandas_gbq                  # Pandas integration with BigQuery
from google.cloud import bigquery  # BigQuery client library
import glob                        # File path pattern matching
import openpyxl                    # Excel file handling
import csv                         # CSV file handling
import re                          # Regular expressions

# Note: The actual imports remain exactly as in the original code

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.4/454.4 kB 8.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for gcloud: filename=gcloud-0.18.3-py3-none-any.whl size=602927 sha256=387b5c8d812c0df2005c5032173a65c26b40443662a1c41b1fd7475f80927b05
  Stored in directory: /root/.cache/pip/wheels/2a/62/75/3d74209bfebb8805823ae74afa28653aa1ea76d8b5a9d741ff
Successfully built gcloud
Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=l5hFefCecDOJEkIYEgHS7YzFDuqZQ1&prompt=consent&token_usage=remote&access_type=offline&code_chal

# Tratamento da planilha PEP para realizar o mergre

In [ ]:
df = pd.read_excel("/content/Pep_ministerios(1).xlsx")

In [ ]:
df.head()

,Ano,Grupo Situação do Vínculo,Orgão Superior,Sexo,Órgão Vinculado,Servidores (anual)
0,Total de Seleção,Total de Seleção,Total de Seleção,Total de Seleção,Total de Seleção,1206091.0
1,2026,Instituidor de pensão,Advocacia-Geral Da Uniao,Fem,AGU,100.0
2,2026,Instituidor de pensão,Advocacia-Geral Da Uniao,Mas,AGU,454.0
3,2026,Instituidor de pensão,Controladoria-Geral Da Uniao,Fem,CGU,25.0
4,2026,Instituidor de pensão,Controladoria-Geral Da Uniao,Mas,CGU,116.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1144 entries, 0 to 1143
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Ano                        1143 non-null   object 
 1   Grupo Situação do Vínculo  1139 non-null   object 
 2   Orgão Superior             1139 non-null   object 
 3   Sexo                       1139 non-null   object 
 4   Órgão Vinculado            1139 non-null   object 
 5   Servidores (anual)         1139 non-null   float64
dtypes: float64(1), object(5)
memory usage: 53.8+ KB


In [ ]:
df['Ano'].unique()

array(['Total de Seleção', 2026, nan, 'Selection Status:',
       'Sem GDF (Servidores): Sim', 'Ano Servidores: 2026',
       'Aba: Servidores'], dtype=object)

In [ ]:
df = df[df['Ano'] == 2026]

In [ ]:
df = df[df['Grupo Situação do Vínculo'] == "Ativo"]

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 416 entries, 364 to 779
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Ano                        416 non-null    object 
 1   Grupo Situação do Vínculo  416 non-null    object 
 2   Orgão Superior             416 non-null    object 
 3   Sexo                       416 non-null    object 
 4   Órgão Vinculado            416 non-null    object 
 5   Servidores (anual)         416 non-null    float64
dtypes: float64(1), object(5)
memory usage: 22.8+ KB


In [ ]:
df.head()

,Ano,Grupo Situação do Vínculo,Orgão Superior,Sexo,Órgão Vinculado,Servidores (anual)
364,2026,Ativo,Advocacia-Geral Da Uniao,Fem,AGU,4169.0
365,2026,Ativo,Advocacia-Geral Da Uniao,Mas,AGU,5136.0
366,2026,Ativo,Controladoria-Geral Da Uniao,Fem,CGU,832.0
367,2026,Ativo,Controladoria-Geral Da Uniao,Mas,CGU,1614.0
368,2026,Ativo,Defensoria Publica Da Uniao,Fem,DPU,373.0


In [ ]:
print(df['Ano'].unique())
print('-------------------------------')
print(df['Grupo Situação do Vínculo'].unique())
print('-------------------------------')
print(df['Orgão Superior'].unique())
print('-------------------------------')
print(df['Sexo'].unique())
print('-------------------------------')
print(df['Órgão Vinculado'].unique())
print('-------------------------------')
print(df['Servidores (anual)'].unique())
print('-------------------------------')


[2026]
-------------------------------
['Ativo']
-------------------------------
['Advocacia-Geral Da Uniao' 'Controladoria-Geral Da Uniao'
 'Defensoria Publica Da Uniao' 'Min Da Integ E Do Desenv Regional'
 'Min Desenv Assis Soci Famil Combate Fome'
 'Min Desenvolv Ind Comercio E Servicos'
 'Min Do Desenv Agr E Agric Familiar' 'Min Dos Dir Hum E Da Cidadania'
 'Min Empreend Microemp Emp Pequeno Porte'
 'Min Gestao E Inov Em Serv Publicos'
 'Minist. Da Justica E Seguranca Publica'
 'Minist. Economia, Fazenda E Planejamento'
 'Ministerio Ciencia Tecnologia Inova'
 'Ministerio Da Agricultura E Pecuaria' 'Ministerio Da Cultura'
 'Ministerio Da Defesa' 'Ministerio Da Educacao' 'Ministerio Da Fazenda'
 'Ministerio Da Igualdade Racial' 'Ministerio Da Infra-Estrutura'
 'Ministerio Da Previdencia Social' 'Ministerio Da Saude'
 'Ministerio Das Cidades' 'Ministerio Das Comunicacoes'
 'Ministerio Das Mulheres' 'Ministerio Das Relacoes Exteriores'
 'Ministerio De Minas E Energia' 'Ministerio De Pe

In [ ]:
# Dicionário de-para: normaliza nomenclatura da lista de dados para a referência
de_para = {
    'Advocacia-Geral Da Uniao': 'Advocacia-Geral da União',
    'Controladoria-Geral Da Uniao': 'Controladoria-Geral da União',
    'Min Da Integ E Do Desenv Regional': 'Ministério da Integração e do Desenvolvimento Regional',
    'Min Desenv Assis Soci Famil Combate Fome': 'Ministério do Desenvolvimento e Assistência Social, Família e Combate à Fome',
    'Min Desenvolv Ind Comercio E Servicos': 'Ministério do Desenvolvimento, Indústria, Comércio e Serviços',
    'Min Do Desenv Agr E Agric Familiar': 'Ministério do Desenvolvimento Agrário e Agricultura Familiar',
    'Min Gestao E Inov Em Serv Publicos': 'Ministério Gestão e Inovação em Serviços Públicos',
    'Minist. Da Justica E Seguranca Publica': 'Ministério da Justiça e Segurança Pública',
    'Minist. Economia, Fazenda E Planejamento': 'Ministério da Fazenda',
    'Ministerio Ciencia Tecnologia Inova': 'Ministério da Ciência, Tecnologia e Inovação',
    'Ministerio Da Agricultura E Pecuaria': 'Ministério da Agricultura e Pecuária',
    'Ministerio Da Cultura': 'Ministério da Cultura',
    'Ministerio Da Defesa': 'Ministério da Defesa',
    'Ministerio Da Economia': 'Ministério da Fazenda',
    'Ministerio Da Educacao': 'Ministério da Educação',
    'Ministerio Da Fazenda': 'Ministério da Fazenda',
    'Ministerio Da Infra-Estrutura': 'Ministério de Portos e Aeroportos',
    'Ministerio Da Previdencia Social': 'Ministério da Previdência Social',
    'Ministerio Da Saude': 'Ministério da Saúde',
    'Ministerio Das Cidades': 'Ministério das Cidades',
    'Ministerio Das Comunicacoes': 'Ministério das Comunicações',
    'Ministerio Das Relacoes Exteriores': 'Ministério das Relações Exteriores',
    'Ministerio De Minas E Energia': 'Ministério De Minas e Energia',
    'Ministerio De Pesca E Aquicultura': 'Ministério da Pesca e Aquicultura',
    'Ministerio De Portos E Aeroportos': 'Ministério de Portos e Aeroportos',
    'Ministerio Do Desenvolvimento Regional': 'Ministério da Integração e do Desenvolvimento Regional',
    'Ministerio Do Esporte': 'Ministério do Esporte',
    'Ministerio Do Meio Ambiente': 'Ministério do Meio Ambiente e Mudança do Clima',
    'Ministerio Do Planej. Desenv. E Gestao': 'Ministério do Planejamento e Orçamento',
    'Ministerio Do Planejamento E Orcamento': 'Ministério do Planejamento e Orçamento',
    'Ministerio Do Trabalho E Emprego': 'Ministério do Trabalho e Emprego',
    'Ministerio Do Turismo': 'Ministério Do Turismo',
    'Ministerio Dos Povos Indigenas': 'Ministério dos Povos Indígenas',
    'Ministerio Dos Transportes': 'Ministério Dos Transportes',
    'Presidencia Da Republica': 'Presidência Da Republica',
    'Secretaria Da Cultura / Pr': 'Ministério da Cultura',
    'Defensoria Publica Da Uniao': 'Defensoria Pública da União',
    'Min Dos Dir Hum E Da Cidadania': 'Ministérios dos Direitos Humanos e da Cidadania',
    'Min Empreend Microemp Emp Pequeno Porte': 'Ministério do Empreendedorismo, da Microempresa e da Empresa de Pequeno Porte',
    'Ministerio Da Igualdade Racial': 'Ministério da Igualdade Racial',
    'Ministerio Das Mulheres': 'Ministério Das Mulheres',
    'Ministerio Do Desenvolvimento Agrario': 'Ministério do Desenvolvimento Agrário e Agricultura Familiar',
    'Ministerio Ind. Com. Exterior E Serviços': 'Ministério do Desenvolvimento, Indústria, Comércio e Serviços',
    'Vice-Presidencia Da Republica': 'Vice-Presidência Da República',
    'Ministerio Da Infraestrutura': 'Ministério de Portos e Aeroportos',
}

# Aplica o mapeamento (ajuste 'orgao' para o nome real da sua coluna)
df['orgao_norm'] = df['Orgão Superior'].map(de_para).fillna(df['Orgão Superior'])

# Verifica o que ficou sem correspondência no dicionário
sem_match = df.loc[~df['Orgão Superior'].isin(de_para), 'Orgão Superior'].unique()
print("Sem correspondência no de_para:", sem_match)

Sem correspondência no de_para: []


In [ ]:
df.head(2)

,Ano,Grupo Situação do Vínculo,Orgão Superior,Sexo,Órgão Vinculado,Servidores (anual),orgao_norm
364,2026,Ativo,Advocacia-Geral Da Uniao,Fem,AGU,4169.0,Advocacia-Geral da União
365,2026,Ativo,Advocacia-Geral Da Uniao,Mas,AGU,5136.0,Advocacia-Geral da União


In [ ]:
df_area.head(2)

NameError: name 'df_area' is not defined

In [ ]:
# df_area vem da planilha que você vai subir (colunas: orgao, area)
df_area = pd.read_excel('/content/Areas Ministerio.xlsx')  # ajuste o caminho/nome

# Limpa espaços nas chaves de ambos para evitar mismatch
df['orgao_norm'] = df['orgao_norm'].str.strip()
df_area['orgao'] = df_area['orgao'].str.strip()

# Traz a área para o df principal usando o ministério como chave
df = df.merge(df_area, how='left', left_on='orgao_norm', right_on='orgao', suffixes=('', '_ref'))

# Confere o que não achou correspondência
print("Sem área:", df.loc[df['area'].isna(), 'orgao_norm'].unique())

Sem área: []


In [ ]:
df.head(2)

,Ano,Grupo Situação do Vínculo,Orgão Superior,Sexo,Órgão Vinculado,Servidores (anual),orgao_norm,orgao,area
0,2026,Ativo,Advocacia-Geral Da Uniao,Fem,AGU,4169.0,Advocacia-Geral da União,Advocacia-Geral da União,"Defesa, segurança e justiça"
1,2026,Ativo,Advocacia-Geral Da Uniao,Mas,AGU,5136.0,Advocacia-Geral da União,Advocacia-Geral da União,"Defesa, segurança e justiça"


In [ ]:
df = df[['Ano', 'orgao_norm', 'area', 'Sexo', 'Servidores (anual)']]

In [ ]:
df.head(2)

,Ano,orgao_norm,area,Sexo,Servidores (anual)
0,2026,Advocacia-Geral da União,"Defesa, segurança e justiça",Fem,4169.0
1,2026,Advocacia-Geral da União,"Defesa, segurança e justiça",Mas,5136.0


In [ ]:
df['Sexo'].unique()

array(['Fem', 'Mas'], dtype=object)

In [ ]:
# Pivota Sexo em colunas, somando os servidores por Ano/orgao_norm/area
df = df.pivot_table(
    index=['Ano', 'orgao_norm', 'area'],
    columns='Sexo',
    values='Servidores (anual)',
    aggfunc='sum'
).reset_index()

df.columns.name = None  # remove o nome 'Sexo' do índice de colunas

In [ ]:
df.head()

,Ano,orgao_norm,area,Fem,Mas
0,2026,Advocacia-Geral da União,"Defesa, segurança e justiça",4169.0,5136.0
1,2026,Controladoria-Geral da União,Regulatória e controle,832.0,1614.0
2,2026,Defensoria Pública da União,"Defesa, segurança e justiça",373.0,283.0
3,2026,Ministério Das Mulheres,Social,117.0,14.0
4,2026,Ministério De Minas e Energia,"Infraestrutura, desenvolvimento econômico e me...",821.0,2054.0


In [ ]:
df = df.rename(columns={
    'Fem': 'feminino',
    'Mas': 'masculino',
    'orgao_norm': 'orgao'
})

In [ ]:
df

,Ano,orgao,area,feminino,masculino
0,2026,Advocacia-Geral da União,"Defesa, segurança e justiça",4169.0,5136.0
1,2026,Controladoria-Geral da União,Regulatória e controle,832.0,1614.0
2,2026,Defensoria Pública da União,"Defesa, segurança e justiça",373.0,283.0
3,2026,Ministério Das Mulheres,Social,117.0,14.0
4,2026,Ministério De Minas e Energia,"Infraestrutura, desenvolvimento econômico e me...",821.0,2054.0
5,2026,Ministério Do Turismo,Social,129.0,109.0
6,2026,Ministério Dos Transportes,"Infraestrutura, desenvolvimento econômico e me...",995.0,2362.0
7,2026,Ministério Gestão e Inovação em Serviços Públicos,"Economia, gestão e planejamento",18389.0,16966.0
8,2026,Ministério da Agricultura e Pecuária,"Infraestrutura, desenvolvimento econômico e me...",2302.0,3971.0
9,2026,"Ministério da Ciência, Tecnologia e Inovação","Infraestrutura, desenvolvimento econômico e me...",2316.0,4422.0


In [ ]:
df['proporcao_fem'] = (df['feminino']/(df['feminino']+df['masculino'])*100).round(2)
df['proporcao_masc'] = (df['masculino']/(df['feminino']+df['masculino'])*100).round(2)
df

,Ano,orgao,area,feminino,masculino,proporcao_fem,proporcao_masc
0,2026,Advocacia-Geral da União,"Defesa, segurança e justiça",4169.0,5136.0,44.80,55.20
1,2026,Controladoria-Geral da União,Regulatória e controle,832.0,1614.0,34.01,65.99
2,2026,Defensoria Pública da União,"Defesa, segurança e justiça",373.0,283.0,56.86,43.14
3,2026,Ministério Das Mulheres,Social,117.0,14.0,89.31,10.69
4,2026,Ministério De Minas e Energia,"Infraestrutura, desenvolvimento econômico e me...",821.0,2054.0,28.56,71.44
5,2026,Ministério Do Turismo,Social,129.0,109.0,54.20,45.80
6,2026,Ministério Dos Transportes,"Infraestrutura, desenvolvimento econômico e me...",995.0,2362.0,29.64,70.36
7,2026,Ministério Gestão e Inovação em Serviços Públicos,"Economia, gestão e planejamento",18389.0,16966.0,52.01,47.99
8,2026,Ministério da Agricultura e Pecuária,"Infraestrutura, desenvolvimento econômico e me...",2302.0,3971.0,36.70,63.30
9,2026,"Ministério da Ciência, Tecnologia e Inovação","Infraestrutura, desenvolvimento econômico e me...",2316.0,4422.0,34.37,65.63


In [ ]:
df_long = (
    df
    .rename(columns={
        'feminino': 'quantidade_fem',
        'masculino': 'quantidade_masc'
    })
    .melt(
        id_vars=['orgao', 'area'],
        value_vars=['quantidade_fem', 'quantidade_masc',
                    'proporcao_fem', 'proporcao_masc'],
        var_name='variavel',
        value_name='valor'
    )
)

df1 = (
    df_long
    .assign(
        genero=lambda x: x['variavel'].str.extract('(fem|masc)'),
        tipo=lambda x: x['variavel'].str.contains('quantidade')
    )
    .pivot_table(
        index=['orgao', 'area', 'genero'],
        columns='tipo',
        values='valor',
        aggfunc='first'
    )
    .reset_index()
    .rename(columns={
        True: 'quantidade',
        False: 'prop_genero'
    })
)


In [ ]:
df1

tipo,orgao,area,genero,prop_genero,quantidade
0,Advocacia-Geral da União,"Defesa, segurança e justiça",fem,44.80,4169.0
1,Advocacia-Geral da União,"Defesa, segurança e justiça",masc,55.20,5136.0
2,Controladoria-Geral da União,Regulatória e controle,fem,34.01,832.0
3,Controladoria-Geral da União,Regulatória e controle,masc,65.99,1614.0
4,Defensoria Pública da União,"Defesa, segurança e justiça",fem,56.86,373.0
...,...,...,...,...,...
67,Ministérios dos Direitos Humanos e da Cidadania,Social,masc,43.66,117.0
68,Presidência Da Republica,"Economia, gestão e planejamento",fem,29.84,1405.0
69,Presidência Da Republica,"Economia, gestão e planejamento",masc,70.16,3304.0
70,Vice-Presidência Da República,"Economia, gestão e planejamento",fem,35.90,28.0


In [ ]:
df1['genero'] = df1['genero'].str.title()
df1['area'] = df1['area'].str.title()
df1['orgao'] = df1['orgao'].str.title()
df1

tipo,orgao,area,genero,prop_genero,quantidade
0,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Fem,44.80,4169.0
1,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Masc,55.20,5136.0
2,Controladoria-Geral Da União,Regulatória E Controle,Fem,34.01,832.0
3,Controladoria-Geral Da União,Regulatória E Controle,Masc,65.99,1614.0
4,Defensoria Pública Da União,"Defesa, Segurança E Justiça",Fem,56.86,373.0
...,...,...,...,...,...
67,Ministérios Dos Direitos Humanos E Da Cidadania,Social,Masc,43.66,117.0
68,Presidência Da Republica,"Economia, Gestão E Planejamento",Fem,29.84,1405.0
69,Presidência Da Republica,"Economia, Gestão E Planejamento",Masc,70.16,3304.0
70,Vice-Presidência Da República,"Economia, Gestão E Planejamento",Fem,35.90,28.0


In [ ]:
def transforme(nome):
    nome =re.sub(r"\bFem\b", "Feminino", nome)
    nome =re.sub(r"\bMasc\b", "Masculino", nome)

    return nome

In [ ]:
df1['genero'] = df1['genero'].apply(transforme)
df1

tipo,orgao,area,genero,prop_genero,quantidade
0,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Feminino,44.80,4169.0
1,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Masculino,55.20,5136.0
2,Controladoria-Geral Da União,Regulatória E Controle,Feminino,34.01,832.0
3,Controladoria-Geral Da União,Regulatória E Controle,Masculino,65.99,1614.0
4,Defensoria Pública Da União,"Defesa, Segurança E Justiça",Feminino,56.86,373.0
...,...,...,...,...,...
67,Ministérios Dos Direitos Humanos E Da Cidadania,Social,Masculino,43.66,117.0
68,Presidência Da Republica,"Economia, Gestão E Planejamento",Feminino,29.84,1405.0
69,Presidência Da Republica,"Economia, Gestão E Planejamento",Masculino,70.16,3304.0
70,Vice-Presidência Da República,"Economia, Gestão E Planejamento",Feminino,35.90,28.0


In [ ]:
df1 = df1.rename(columns={'quantidade':'quantidade_vinculos'})

In [ ]:
df1 = df1[['orgao', 'area', 'genero', 'prop_genero', 'quantidade_vinculos']]
df1

tipo,orgao,area,genero,prop_genero,quantidade_vinculos
0,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Feminino,44.80,4169.0
1,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Masculino,55.20,5136.0
2,Controladoria-Geral Da União,Regulatória E Controle,Feminino,34.01,832.0
3,Controladoria-Geral Da União,Regulatória E Controle,Masculino,65.99,1614.0
4,Defensoria Pública Da União,"Defesa, Segurança E Justiça",Feminino,56.86,373.0
...,...,...,...,...,...
67,Ministérios Dos Direitos Humanos E Da Cidadania,Social,Masculino,43.66,117.0
68,Presidência Da Republica,"Economia, Gestão E Planejamento",Feminino,29.84,1405.0
69,Presidência Da Republica,"Economia, Gestão E Planejamento",Masculino,70.16,3304.0
70,Vice-Presidência Da República,"Economia, Gestão E Planejamento",Feminino,35.90,28.0


In [ ]:
df1

tipo,orgao,area,genero,prop_genero,quantidade_vinculos
0,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Feminino,44.80,4169.0
1,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Masculino,55.20,5136.0
2,Controladoria-Geral Da União,Regulatória E Controle,Feminino,34.01,832.0
3,Controladoria-Geral Da União,Regulatória E Controle,Masculino,65.99,1614.0
4,Defensoria Pública Da União,"Defesa, Segurança E Justiça",Feminino,56.86,373.0
...,...,...,...,...,...
67,Ministérios Dos Direitos Humanos E Da Cidadania,Social,Masculino,43.66,117.0
68,Presidência Da Republica,"Economia, Gestão E Planejamento",Feminino,29.84,1405.0
69,Presidência Da Republica,"Economia, Gestão E Planejamento",Masculino,70.16,3304.0
70,Vice-Presidência Da República,"Economia, Gestão E Planejamento",Feminino,35.90,28.0


In [ ]:
df1 = df1[['orgao', 'area', 'genero', 'prop_genero', 'quantidade_vinculos']]

In [ ]:
df1['quantidade_vinculos'] = df1['quantidade_vinculos'].astype(int)
df1

tipo,orgao,area,genero,prop_genero,quantidade_vinculos
0,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Feminino,44.80,4169
1,Advocacia-Geral Da União,"Defesa, Segurança E Justiça",Masculino,55.20,5136
2,Controladoria-Geral Da União,Regulatória E Controle,Feminino,34.01,832
3,Controladoria-Geral Da União,Regulatória E Controle,Masculino,65.99,1614
4,Defensoria Pública Da União,"Defesa, Segurança E Justiça",Feminino,56.86,373
...,...,...,...,...,...
67,Ministérios Dos Direitos Humanos E Da Cidadania,Social,Masculino,43.66,117
68,Presidência Da Republica,"Economia, Gestão E Planejamento",Feminino,29.84,1405
69,Presidência Da Republica,"Economia, Gestão E Planejamento",Masculino,70.16,3304
70,Vice-Presidência Da República,"Economia, Gestão E Planejamento",Feminino,35.90,28


# Upload

In [ ]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   orgao                72 non-null     object 
 1   area                 72 non-null     object 
 2   genero               72 non-null     object 
 3   prop_genero          72 non-null     float64
 4   quantidade_vinculos  72 non-null     int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 2.9+ KB


# Envian do para o GBQ

In [ ]:
# Import the bigquery library from google.cloud
from google.cloud import bigquery

# Initialize the BigQuery client, specifying the Google Cloud project ID.
# This client object is the main entry point for interacting with the BigQuery API.
client = bigquery.Client(project='repositoriodedadosgpsp')

# Create a reference to the BigQuery dataset named 'perfil_remuneracao'.
# This object points to the dataset where the table will be created or updated.
dataset_ref = client.dataset('perfil_remuneracao')

# Define the schema for the destination BigQuery table.
# The schema is a list of SchemaField objects, where each object defines a column's:
# 1. Name (e.g., 'ano')
# 2. Data type (e.g., 'INTEGER')
# 3. Description (e.g., 'Ano de referencia da informacao')
schema = [bigquery.SchemaField('orgao', 'STRING', description= 'Órgão'),
          bigquery.SchemaField('area', 'STRING', description= 'Agrupamento de áreas do órgão'),
          bigquery.SchemaField('genero', 'STRING', description= 'Gênero autodeclarado ou não'),
          bigquery.SchemaField('quantidade_vinculos', 'INTEGER', description= 'Número total de vinculos observados'),
          bigquery.SchemaField('prop_genero', 'FLOAT', description= 'Proporção de vínculos por genero'),
          ]

# Create a reference to the target table within the dataset specified earlier.
# The table will be named 'CNES_medicos_mil_habitantes_v1'.
table_ref = dataset_ref.table('PEP_ministerio_genero_v2') # table name in the format SOURCE_intuitive_data_name

# Configure the load job by creating a LoadJobConfig object.
# Here, we specify the schema that BigQuery should use for the table. This ensures
# that the columns in BigQuery have the correct data types and descriptions.
job_config = bigquery.LoadJobConfig(schema=schema)

# Start the job to load data from the pandas DataFrame 'df' into the specified BigQuery table ('table_ref').
# The job is configured with the previously defined 'job_config'. This command sends the data to BigQuery.
job = client.load_table_from_dataframe(df1, table_ref, job_config=job_config)

# Wait for the load job to complete and retrieve its result.
# This line is blocking and will pause the script's execution until the data upload is finished.
# It's crucial for ensuring the data is fully loaded before the script ends or proceeds.
job.result()

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


LoadJob<project=repositoriodedadosgpsp, location=US, id=81bc25a4-c99c-4fcb-aabf-97abb36dfa64>